# Cleaned_data (ingredient_merged2) 기준으로 청크화 하기

### 청크 변환 파이프라인 — ingredient_merged2.json 기준
- coos_score: 이미 1/2/3으로 정제됨 (기존 [안전]/[주의]/[위험] 텍스트 아님)
- 없어진 컬럼: coos_cas_no, coos_country, hw_ingredient_id, hw_is_allergy, hw_purposes
- product_db.csv: 별도 파일로 이미 존재 (이 스크립트에서 재생성하지 않음)
- 중복 제거: 직접 확인 후 처리할 수 있도록 중복 현황 리포트 출력

In [113]:
import json
import numpy as np
import re
from collections import Counter, defaultdict

# 1. 가중치 설정 (파트너 코드와 동일한 1.0 통일 또는 기존 프리셋 사용)
WEIGHT_PRESETS = {
    1: {"ewg": 0.33, "basic_info": 0.33, "expert": 0.33, "review": 0.33},
    4: {"ewg": 0.45, "basic_info": 0.45, "expert": 0.10, "review": 0.10}, # 임의 추가
}
selected_preset = WEIGHT_PRESETS[4]

# 2. 데이터 경로 설정 및 로드
BASE_PATH = "/content"
INPUT_FILE = f"{BASE_PATH}/ingredient_merged2.json"

with open(INPUT_FILE, "rb") as f:
    raw = f.read()

# BOM 제거 및 파싱
if raw[:3] == b'\xef\xbb\xbf':
    raw = raw[3:]
data = json.loads(raw.decode("utf-8", errors="ignore"))

print(f"초기 로드된 총 행 수: {len(data)}")

초기 로드된 총 행 수: 87712


In [114]:
# 1. 설정

WEIGHT_PRESETS = {
    1: {"ewg": 0.33, "basic_info": 0.33, "expert": 0.33},   # 균등
    2: {"ewg": 0.50, "basic_info": 0.35, "expert": 0.15},   # EWG 중심
    3: {"ewg": 0.40, "basic_info": 0.45, "expert": 0.15},   # 기본정보 중심
    4: {"ewg": 0.45, "basic_info": 0.45, "expert": 0.10},   # 균형
}

In [115]:
BASE_PATH = "/content"
INPUT_FILE = f"{BASE_PATH}/ingredient_merged2.json"

with open(INPUT_FILE, "rb") as f:
    raw = f.read()
if raw[:3] == b'\xef\xbb\xbf':
    raw = raw[3:]
text = raw.decode("utf-8", errors="ignore")
data = json.loads(text)

print(f"총 행 수: {len(data)}")
print(f"컬럼 목록: {list(data[0].keys())}")

총 행 수: 87712
컬럼 목록: ['ingredient_ko', 'ingredient_en', 'pc_rating', 'pc_effect', 'pc_category', 'pc_description', 'coos_function', 'coos_type', 'coos_kr_restricted', 'coos_cn_restricted', 'coos_tw_restricted', 'coos_jp_restricted', 'coos_eu_restricted', 'coos_asean_restricted', 'coos_ai_description', 'coos_score', 'coos_data_grade', 'hw_product_id', 'hw_product_name', 'hw_brand_name', 'hw_avg_ratings', 'hw_review_count', 'hw_price', 'hw_consumer_price', 'hw_primary_attr', 'hw_topics_positive', 'hw_topics_negative', 'hw_ingredient_count', 'hw_ewg', 'hw_ewg_data_availability_text', 'hw_purpose', 'hw_limitation', 'hw_forbidden', 'hw_category']


In [116]:
seen = set()
unique_data = []

for row in data:
    key = (
        str(row.get("ingredient_ko", "")).strip(),
        str(row.get("hw_product_id", "")).strip(),
    )
    if key not in seen:
        seen.add(key)
        unique_data.append(row)

print(f"중복 제거: {len(data)}행 -> {len(unique_data)}행 ({len(data) - len(unique_data)}개 제거)")
data = unique_data

중복 제거: 87712행 -> 69863행 (17849개 제거)


In [117]:
# 1. 범위형 등급("1_2", "1~3" 등) 마지막 숫자만 추출 (파트너 코드 반영 부분)
def extract_max_rating(rating_val):
    if rating_val is None:
        return None
    s = str(rating_val).strip()
    parts = re.split(r'[_~-]', s)
    if len(parts) > 1:
        return parts[-1].strip()
    return s

# 2. 유효값 판별 함수
def is_valid_value(val) -> bool:
    if val is None:
        return False
    s = str(val).strip()
    if s in ("", "nan", "NaN", "None", "없음", "0"):
        return False
    try:
        if np.isnan(float(s)):
            return False
    except (ValueError, TypeError):
        pass
    return True

# 3. 전체 데이터 변환 적용
for row in data:
    # 범위형 등급 변환
    for col in ["coos_score", "hw_ewg", "pc_rating"]:
        if row.get(col):
            row[col] = extract_max_rating(row[col])

    # 빈 값을 0으로 일괄 변환
    for col in ["coos_score", "hw_ewg", "pc_rating"]:
        if row.get(col) is None or str(row.get(col, "")).strip() in ("", "nan", "NaN", "None"):
            row[col] = 0

print("범위형 등급 단일 숫자 변환 및 결측치(0) 처리 완료")

범위형 등급 단일 숫자 변환 및 결측치(0) 처리 완료


In [118]:
def generate_chunks_ingredient_only(all_data, current_weights):
    ingredient_groups = defaultdict(list)
    for row in all_data:
        ing_name = str(row.get("ingredient_ko", "")).strip()
        if not ing_name:
            continue
        ingredient_groups[ing_name].append(row)

    all_chunks = []

    for ing, rows in ingredient_groups.items():
        ingredient_en = ""
        for r in rows:
            if is_valid_value(r.get("ingredient_en")):
                ingredient_en = str(r["ingredient_en"]).strip()
                break

        def collect_unique(col):
            vals = set()
            for r in rows:
                if is_valid_value(r.get(col)):
                    vals.add(str(r[col]).strip())
            return vals

        # 메타데이터용 고유값 전체 수집
        coos_score_vals = list(collect_unique("coos_score"))
        hw_ewg_vals = list(collect_unique("hw_ewg"))
        pc_rating_vals = list(collect_unique("pc_rating"))

        base_meta = {
            "ingredient_ko": ing,
            "ingredient_en": ingredient_en,
            "coos_score": ", ".join(sorted(coos_score_vals)) if coos_score_vals else 0,
            "hw_ewg": ", ".join(sorted(hw_ewg_vals)) if hw_ewg_vals else 0,
            "pc_rating": ", ".join(sorted(pc_rating_vals)) if pc_rating_vals else 0,
        }

        # 1) EWG 청크
        ewg_parts = []
        if coos_score_vals:
            score_map = {"1": "안전", "2": "주의", "3": "위험"}
            labels = [f"{score_map.get(s, s)}({s}등급)" for s in sorted(coos_score_vals)]
            ewg_parts.append(f"EWG 스코어: {', '.join(labels)}")

        for col, label in [
            ("hw_ewg", "화해 EWG"),
            ("hw_ewg_data_availability_text", "EWG 데이터 가용성"),
            ("coos_data_grade", "coos 데이터 등급"),
            ("pc_rating", "폴라초이스 등급")
        ]:
            vals = collect_unique(col)
            if vals:
                ewg_parts.append(f"{label}: {', '.join(sorted(vals))}")

        if ewg_parts:
            all_chunks.append({
                "page_content": f"[{ing}] " + " / ".join(ewg_parts),
                "metadata": {**base_meta, "chunk_type": "ewg", "chunk_weight": current_weights["ewg"]}
            })

        # 2) Basic Info 청크
        basic_parts = []
        for col, label in [
            ("pc_effect", "효과"), ("pc_category", "분류"),
            ("coos_function", "기능"), ("coos_type", "종류"),
            ("hw_purpose", "용도"), ("hw_limitation", "사용제한"), ("hw_forbidden", "사용금지")
        ]:
            vals = collect_unique(col)
            if vals:
                basic_parts.append(f"{label}: {' | '.join(sorted(vals))}")

        if basic_parts:
            all_chunks.append({
                "page_content": f"[{ing}] " + " / ".join(basic_parts),
                "metadata": {**base_meta, "chunk_type": "basic_info", "chunk_weight": current_weights["basic_info"]}
            })

        # 3) Expert 청크
        expert_parts = []
        for col, label in [
            ("pc_description", "성분설명"), ("coos_ai_description", "AI설명"),
            ("coos_kr_restricted", "국내허용"), ("coos_cn_restricted", "중국허용"),
            ("coos_tw_restricted", "대만허용"), ("coos_jp_restricted", "일본허용"),
            ("coos_eu_restricted", "독일허용"), ("coos_asean_restricted", "베트남허용"),
            ("hw_category", "카테고리")
        ]:
            vals = collect_unique(col)
            if vals:
                expert_parts.append(f"{label}: {' | '.join(sorted(vals))}")

        if expert_parts:
            all_chunks.append({
                "page_content": f"[{ing}] " + " | ".join(expert_parts),
                "metadata": {**base_meta, "chunk_type": "expert", "chunk_weight": current_weights["expert"]}
            })

    return all_chunks
print("청크 생성 함수 정의 완료")

청크 생성 함수 정의 완료


In [119]:
# 프리셋별 청크 생성 및 개별 JSON 파일 저장
for preset_id, current_weights in WEIGHT_PRESETS.items():
    print(f"\n[Preset {preset_id}] 가중치({current_weights}) 적용 청크 생성 시작...")

    # 청크 생성
    final_chunks = generate_chunks_ingredient_only(data, current_weights)

    # 동적 파일명 지정 및 저장
    output_file = f"{BASE_PATH}/ingredient_chunks_preset_{preset_id}.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(final_chunks, f, ensure_ascii=False, indent=2)

    # 각 프리셋별 결과 출력
    type_counts = Counter(c["metadata"]["chunk_type"] for c in final_chunks)
    print(f" -> 총 {len(final_chunks)}개 청크 저장 완료: {output_file}")
    print(f" -> 분포: {dict(type_counts)}")

print("\n✅ 모든 프리셋 파일 생성 및 저장 완료")


[Preset 1] 가중치({'ewg': 0.33, 'basic_info': 0.33, 'expert': 0.33}) 적용 청크 생성 시작...
 -> 총 42227개 청크 저장 완료: /content/ingredient_chunks_preset_1.json
 -> 분포: {'basic_info': 17144, 'expert': 17144, 'ewg': 7939}

[Preset 2] 가중치({'ewg': 0.5, 'basic_info': 0.35, 'expert': 0.15}) 적용 청크 생성 시작...
 -> 총 42227개 청크 저장 완료: /content/ingredient_chunks_preset_2.json
 -> 분포: {'basic_info': 17144, 'expert': 17144, 'ewg': 7939}

[Preset 3] 가중치({'ewg': 0.4, 'basic_info': 0.45, 'expert': 0.15}) 적용 청크 생성 시작...
 -> 총 42227개 청크 저장 완료: /content/ingredient_chunks_preset_3.json
 -> 분포: {'basic_info': 17144, 'expert': 17144, 'ewg': 7939}

[Preset 4] 가중치({'ewg': 0.45, 'basic_info': 0.45, 'expert': 0.1}) 적용 청크 생성 시작...
 -> 총 42227개 청크 저장 완료: /content/ingredient_chunks_preset_4.json
 -> 분포: {'basic_info': 17144, 'expert': 17144, 'ewg': 7939}

✅ 모든 프리셋 파일 생성 및 저장 완료


In [121]:
# 9. 검증 (생성된 4개 파일 전수 조사)
import random
import os

print("══ 파이프라인 생성 결과 검증 ══")

# 앞서 정의한 저장 경로와 일치해야 합니다.
for p_idx in range(1, 5):
    # 파일명 규칙 확인: preset1, preset2...
    file_path = f"{BASE_PATH}/ingredient_chunks_preset_{p_idx}.json"

    if not os.path.exists(file_path):
        print(f"\n❌ 파일 없음: {file_path}")
        continue

    with open(file_path, "r", encoding="utf-8") as f:
        check_chunks = json.load(f)

    print(f"\n--- [프리셋 {p_idx}번 파일 검증] ---")
    print(f"1) 경로: {file_path}")
    print(f"2) 총 청크 수: {len(check_chunks)}")

    # 타입별 개수 및 가중치 확인
    type_stats = defaultdict(list)
    for c in check_chunks:
        m = c.get("metadata", {})
        type_stats[m.get("chunk_type")].append(m.get("chunk_weight", 0))

    print(f"3) 타입별 분포 및 가중치:")
    for t in ["ewg", "basic_info", "expert"]:
        weights = type_stats.get(t, [])
        if weights:
            avg_w = sum(weights) / len(weights)
            print(f"   - {t:12}: {len(weights):5}개 (Weight: {avg_w:.2f})")
        else:
            print(f"   - {t:12}: 데이터 없음")

    # 4) 중복성 검사 - 성분명 + 제품ID 조합으로 체크
    print(f"4) 중복성 검사:")
    for chunk_type in ["ewg", "basic_info", "expert"]:
        keys = [
            (c["metadata"]["ingredient_ko"], str(c["metadata"].get("hw_product_id", "")))
            for c in check_chunks if c["metadata"]["chunk_type"] == chunk_type
        ]
        dup = {k: v for k, v in Counter(keys).items() if v > 1}
        if dup:
            print(f"   ❌ {chunk_type:10} 중복 발견! ({len(dup)}건)")
            for k, v in list(dup.items())[:3]:
                print(f"      예: {k[0]} / product_id={k[1]} → {v}회")
        else:
            print(f"   ✅ {chunk_type:10} 중복 없음")

    # 5) 빈 내용 및 샘플 검사
    empty = [c for c in check_chunks if not str(c.get("page_content", "")).strip()]
    if empty:
        print(f"5) ❌ 빈 청크 발견: {len(empty)}개")
    else:
        sample = random.choice(check_chunks)
        print(f"5) ✅ 내용 이상 없음")
        print(f"   [샘플] {sample['metadata']['chunk_type']}: {sample['page_content'][:60]}...")

print("\n✨ 모든 파일 검증 프로세스가 완료되었습니다.")

══ 파이프라인 생성 결과 검증 ══

--- [프리셋 1번 파일 검증] ---
1) 경로: /content/ingredient_chunks_preset_1.json
2) 총 청크 수: 42227
3) 타입별 분포 및 가중치:
   - ewg         :  7939개 (Weight: 0.33)
   - basic_info  : 17144개 (Weight: 0.33)
   - expert      : 17144개 (Weight: 0.33)
4) 중복성 검사:
   ✅ ewg        중복 없음
   ✅ basic_info 중복 없음
   ✅ expert     중복 없음
5) ✅ 내용 이상 없음
   [샘플] ewg: [애플민트잎추출물] EWG 스코어: 안전(1등급) / 화해 EWG: 1 / EWG 데이터 가용성: 데이터 등...

--- [프리셋 2번 파일 검증] ---
1) 경로: /content/ingredient_chunks_preset_2.json
2) 총 청크 수: 42227
3) 타입별 분포 및 가중치:
   - ewg         :  7939개 (Weight: 0.50)
   - basic_info  : 17144개 (Weight: 0.35)
   - expert      : 17144개 (Weight: 0.15)
4) 중복성 검사:
   ✅ ewg        중복 없음
   ✅ basic_info 중복 없음
   ✅ expert     중복 없음
5) ✅ 내용 이상 없음
   [샘플] basic_info: [참나리추출물] 기능: AI Conditioning AI Soothing/Calming EU SKIN CON...

--- [프리셋 3번 파일 검증] ---
1) 경로: /content/ingredient_chunks_preset_3.json
2) 총 청크 수: 42227
3) 타입별 분포 및 가중치:
   - ewg         :  7939개 (Weight: 0.40)
   - basic_info  : 17144개 (Weig